# 7차시 정답 Notebook — 설비·공정 비교와 수율·불량 분석

## 1단계. 데이터 읽고 설비별 평균 온도 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week07/week07_equipment_yield.csv")
print(df.groupby("설비번호")["온도_섭씨"].mean())

설비번호
EQ-01    300.205769
EQ-02    300.065385
EQ-03    299.346429
EQ-04    301.651136
Name: 온도_섭씨, dtype: float64


## 2단계. 설비별 합격률 비교하기

In [2]:
pass_rate = df.groupby("설비번호")["합격여부"].apply(lambda s: (s == 1).mean() * 100)
print(pass_rate)

설비번호
EQ-01    98.076923
EQ-02    86.538462
EQ-03    91.071429
EQ-04    75.000000
Name: 합격여부, dtype: float64


## 3단계. 로트별 수율 계산하기

In [3]:
lot_summary = df.groupby("로트번호").agg(
    설비번호=("설비번호", "first"),
    검사수량=("합격여부", "count"),
    양품수량=("합격여부", lambda s: (s == 1).sum()),
)
lot_summary["수율_pct"] = (lot_summary["양품수량"] / lot_summary["검사수량"] * 100).round(1)

print(lot_summary.head())

           설비번호  검사수량  양품수량  수율_pct
로트번호                               
LOT-0001  EQ-01     4     4   100.0
LOT-0002  EQ-01     4     4   100.0
LOT-0003  EQ-01     4     4   100.0
LOT-0004  EQ-01     4     4   100.0
LOT-0005  EQ-04     4     4   100.0


## 4단계. 저수율 로트 찾기

In [4]:
low_yield_lots = lot_summary[lot_summary["수율_pct"] < 80]
print(len(low_yield_lots), "개 로트가 수율 80% 미만")
print(low_yield_lots["설비번호"].value_counts())

21 개 로트가 수율 80% 미만
설비번호
EQ-04    11
EQ-03     5
EQ-02     4
EQ-01     1
Name: count, dtype: int64


## 5단계. 불량 유형 파레토

In [5]:
defect_counts = df[df["합격여부"] == -1]["불량유형"].value_counts()
print(defect_counts)

불량유형
두께 불량        11
패턴 불량         6
파티클           5
오염            5
정렬 불량         4
식각 부족         4
전기적 특성 불량     1
Name: count, dtype: int64


## 6단계(종합). 결론 정리하기

In [6]:
worst_equipment = pass_rate.idxmin()
print(f"합격률이 가장 낮은 설비: {worst_equipment}")

합격률이 가장 낮은 설비: EQ-04


## 7단계. 오늘의 학습을 한 문장으로 정리하기

> groupby로 설비별 평균·합격률을 비교하고, 로트별 수율을 따로 계산하면 전체 평균만으로는 보이지 않던 문제 로트를 찾을 수 있다.

## 8단계. AI에게 질문하며 더 알아보기
오늘 배운 groupby 집계에 대해 AI 챗봇에게 질문해 이해를 넓혀봅니다.

질문 예시:
- "groupby()와 pivot_table()은 어떻게 다른가요?"
- "파레토 법칙(80/20 법칙)이 정확히 무엇인가요?"
- "agg()에서 그룹마다 서로 다른 열에 서로 다른 함수를 적용하려면 어떻게 하나요?"
- "수율(yield)과 합격률은 같은 개념인가요, 다른 개념인가요?"

## 9단계. AI에게 코드 생성 요청하고 직접 실행해보기
프롬프트 예시: "설비별 합격률이 담긴 데이터를 받아서, 90% 미만인 설비만 경고 문구와 함께
출력하는 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드와 실행 결과입니다(앞서 계산한 `pass_rate`를 그대로 사용합니다).

In [ ]:
for eq, rate in pass_rate.items():
    if rate < 90:
        print(f"⚠️ {eq} 합격률 {rate:.1f}% → 점검 필요")
    else:
        print(f"✅ {eq} 합격률 {rate:.1f}% → 정상")